# Sanidade: fontes oficiais × JSON publicado
Execute de cima para baixo. O notebook faz somente **GETs públicos** e grava cache/diagnósticos locais; não publica no bucket nem executa o job.

Há duas evidências distintas: **checagem independente** dos CSVs (seções, eleitores, locais e votos) e **reprodução do cruzamento** usando funções do pipeline. A segunda pode reproduzir um bug compartilhado e não substitui a primeira.

As fontes TSE são mutáveis. Comparamos SHA-256 dos downloads com os hashes do snapshot; diferença de versão não deve ser confundida com erro do ETL. O modo padrão também baixa as cópias arquivadas da extração quando disponíveis.

Os quatro ZIPs somam aproximadamente **500 MB por conjunto**; modo `ambos` pode baixar cerca de 1 GB. CSVs são lidos em streaming. Não abrimos o CSV final local. Não há validação de extrações perdidas.

In [1]:
# 1. Bibliotecas, projeto e utilitários de exibição
import csv
import hashlib
import html
import io
import json
import math
import re
import sys
import unicodedata
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, urlsplit, urlunsplit, parse_qsl, urlencode
from urllib.request import Request, urlopen
from uuid import uuid4
from IPython.display import HTML, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pipeline-config.json').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts import gerar_dados as pipeline
from scripts.storage_io import typed_row

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
CACHE = ROOT / '.cache' / 'sanidade'
OUTPUT = CACHE / 'relatorios' / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=True)

def table(records, limit=20):
    records = list(records)
    if not records:
        print('(sem registros)')
        return
    columns = list(dict.fromkeys(k for r in records[:limit] for k in r))
    def text(value):
        return html.escape(str(value))
    body = ''.join('<tr>' + ''.join(f'<td>{text(r.get(c, ""))}</td>' for c in columns) + '</tr>' for r in records[:limit])
    display(HTML('<table><thead><tr>' + ''.join(f'<th>{text(c)}</th>' for c in columns) + '</tr></thead><tbody>' + body + '</tbody></table>'))
    print(f'{len(records)} registro(s); exibidos até {limit}.')

print('Projeto:', ROOT)
print('Python:', sys.version.split()[0])
print('Saída local:', OUTPUT)

Projeto: /Users/Vinicius/repos/secoes_2T2026
Python: 3.14.2
Saída local: /Users/Vinicius/repos/secoes_2T2026/.cache/sanidade/relatorios/20261021T175212280869Z


In [2]:
# 2. Parâmetros — edite aqui e execute novamente as células seguintes
CONFIG = json.loads((ROOT / 'pipeline-config.json').read_text())
DASHBOARD = json.loads((ROOT / 'dashboard-config.json').read_text())
JSON_URL = 'https://storage.googleapis.com/' + CONFIG['storage']['bucket'] + '/' + quote(CONFIG['storage']['object'], safe='/')
SNAPSHOT_AT = None  # None = extração cronologicamente mais recente; ou ISO completo
MODO = 'ambos'  # 'atuais', 'arquivadas' ou 'ambos'
ATUALIZAR_DOWNLOADS = True  # True = GET novo das fontes atuais; False = cache local
FONTES_TSE = ('changes', 'registry', 'votes', 'details')
SECAO_INVESTIGAR = ('PR', 171, 110)  # UF, zona, seção; município opcional na investigação
LIMITE_AMOSTRAS = 10
assert MODO in {'atuais', 'arquivadas', 'ambos'}

def fresh_url(url):
    parts = urlsplit(url)
    query = parse_qsl(parts.query, keep_blank_values=True)
    query.append(('_sanidade', uuid4().hex))
    return urlunsplit((parts.scheme, parts.netloc, parts.path, urlencode(query), parts.fragment))

# O JSON publicado é lido uma única vez e congelado para todas as etapas.
request = Request(fresh_url(JSON_URL), headers={'Cache-Control': 'no-cache', 'Accept-Encoding': 'identity'})
with urlopen(request, timeout=120) as response:
    JSON_BYTES = response.read()
    JSON_HEADERS = dict(response.headers)
PAYLOAD = json.loads(JSON_BYTES)
(OUTPUT / 'json_referencia.json').write_bytes(JSON_BYTES)
print('JSON:', JSON_URL)
print('SHA-256 JSON:', hashlib.sha256(JSON_BYTES).hexdigest())
print('Modo:', MODO)
table([{'fonte': n, 'url_atual': CONFIG['sources'][n]} for n in FONTES_TSE])

JSON: https://storage.googleapis.com/extracao_alteracao_locais_votacao/secoes_alteradas_2t_2026.json
SHA-256 JSON: 43ec45cb7ccf92f3dd4c26c28d96f0e00afa4f811a6135e79c0553391d0add17
Modo: ambos


fonte,url_atual
changes,https://cdn.tse.jus.br/estatistica/sead/odsele/arquivos_gerados/eleitorado_eleicao_2026/20261019_105000/71d5e6b06fa299337b05541127926cd7/secoes_eleitorais_alteradas_2t-pais_2026.csv.zip
registry,https://cdn.tse.jus.br/estatistica/sead/odsele/eleitorado_locais_votacao/eleitorado_local_votacao_2026.zip
votes,https://cdn.tse.jus.br/estatistica/sead/odsele/votacao_secao/votacao_secao_2026_BR.zip
details,https://cdn.tse.jus.br/estatistica/sead/odsele/detalhe_votacao_secao/detalhe_votacao_secao_2026.zip


4 registro(s); exibidos até 20.


In [3]:
# 3. Estrutura e seleção do JSON de referência
assert isinstance(PAYLOAD, dict) and isinstance(PAYLOAD.get('snapshots'), list) and PAYLOAD['snapshots'], 'Histórico ausente/vazio'
def instant(value):
    parsed = datetime.fromisoformat(value.replace('Z', '+00:00'))
    assert parsed.utcoffset() is not None, 'Data sem fuso'
    return parsed.astimezone(timezone.utc)

SNAPSHOTS = sorted(PAYLOAD['snapshots'], key=lambda s: instant(s['extracted_at']))
SNAPSHOT = SNAPSHOTS[-1] if SNAPSHOT_AT is None else next(s for s in SNAPSHOTS if instant(s['extracted_at']) == instant(SNAPSHOT_AT))
JSON_ROWS = SNAPSHOT['rows']
assert isinstance(JSON_ROWS, list) and JSON_ROWS, 'Extração sem linhas'
VERSOES_SNAPSHOT = SNAPSHOT.get('sources', {})
HISTORICO = [{'extracted_at': s['extracted_at'], 'secoes': len(s['rows']),
              'eleitores': sum(r['eleitores'] for r in s['rows']),
              'fontes_arquivadas': sum(bool(v.get('archived')) for v in s.get('sources', {}).values())} for s in SNAPSHOTS]
table(HISTORICO, limit=100)
print('Selecionada:', SNAPSHOT['extracted_at'])
table(JSON_ROWS, limit=3)
if not VERSOES_SNAPSHOT:
    print('ATENÇÃO: snapshot sem proveniência; não é possível afirmar igualdade de versão.')

extracted_at,secoes,eleitores,fontes_arquivadas
2026-10-19T18:48:21.245092+00:00,774,229018,0
2026-10-19T18:53:33.863224+00:00,774,229018,0
2026-10-19T23:33:48.847590+00:00,774,229018,0
2026-10-20T09:33:59.101377+00:00,774,229018,0
2026-10-20T14:33:39.887716+00:00,774,229018,0
2026-10-20T18:13:11.907323+00:00,774,229018,0
2026-10-20T18:33:05.419297+00:00,774,229018,0
2026-10-20T18:41:06.106778+00:00,774,229018,1
2026-10-21T17:44:31.855417+00:00,774,229018,4


9 registro(s); exibidos até 100.
Selecionada: 2026-10-21T17:44:31.855417+00:00


uf,municipio,cd_ibge,zona,secao,eleitores,local_1t,endereco_1t,lat_1t,lon_1t,local_2t,endereco_2t,bairro_2t,lat_2t,lon_2t,distancia_km,regiao_longe_da_urna,aptos_1t,abstencao_1t,lula_1t,flavio_1t,outros_1t,brancos_1t,nulos_1t
AC,BRASILÉIA,1200104,6,54,233,ESCOLA FONTENELE DE CASTRO,AV. DR. MANOEL MARINHO MONTE N. 019,-11.00020931,-68.75054031,CAMPUS FRONTEIRA DO ALTO ACRE - UFAC,"BR 317, KM 09, SENTIDO ASSIS BRASIL",None,None,None,None,None,234,41,86,94,5,2,6
AC,BRASILÉIA,1200104,6,113,255,ESCOLA FONTENELE DE CASTRO,AV. DR. MANOEL MARINHO MONTE N. 019,-11.00020931,-68.75054031,ESCOLA ANTÔNIO CARNEIRO DA SILVA - KM 04 POLO AGROFLORESTAL,BR 317 KM 04 RML DO POLO KM 04,None,None,None,None,None,255,56,65,125,6,2,1
AC,BRASILÉIA,1200104,6,146,272,ESCOLA MENINO JESUS,"RUA JOÃO JOVINO DE OLIVEIRA, N. 300",-11.00550299,-68.75489194,UNIDADE BÁSICA DE SAÚDE SIMÃO MANSOUR BARTHA,"RUA COSMO SOUZA CAVALCANTE, S/N, ALBERTO CASTRO",None,None,None,None,None,273,63,33,156,16,3,2


774 registro(s); exibidos até 3.


In [ ]:
# Descoberta da URL vigente antes dos downloads da etapa seguinte.
# Em Jupyter, o Playwright síncrono roda em outra thread (há event loop ativo).
# Falha do portal interrompe esta célula: nunca substituir pelo link do snapshot antigo.
import asyncio
from scripts.tse_source import discover_changes

CONFIG = json.loads((ROOT / 'pipeline-config.json').read_text())
if CONFIG.get('changesPage') and MODO in {'atuais', 'ambos'}:
    DESCOBERTA = await asyncio.to_thread(discover_changes, CONFIG['changesPage'], CONFIG['year'])
    CONFIG['sources']['changes'] = DESCOBERTA['url']
    print('Fonte vigente:', DESCOBERTA)
else:
    print('Modo apenas arquivadas ou fonte explícita: não há descoberta de fonte atual.')

In [4]:
# 4. Downloads diretos + verificação dos bytes (não confiamos apenas no sidecar)
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

PATHS = {}
VERSOES = []
FALHAS_DOWNLOAD = []
for modo in (['atuais', 'arquivadas'] if MODO == 'ambos' else [MODO]):
    PATHS[modo] = {}
    for name in FONTES_TSE:
        published = VERSOES_SNAPSHOT.get(name, {})
        archived = published.get('archived')
        if modo == 'arquivadas' and not archived:
            FALHAS_DOWNLOAD.append({'modo': modo, 'fonte': name, 'erro': 'Snapshot sem cópia arquivada'})
            continue
        url = CONFIG['sources'][name] if modo == 'atuais' else 'https://storage.googleapis.com/' + quote(archived.removeprefix('gs://'), safe='/')
        print('Baixando/lendo:', modo, name, flush=True)
        try:
            path = pipeline.download(url, CACHE / modo, refresh=(modo == 'atuais' and ATUALIZAR_DOWNLOADS))
            digest = sha256_file(path)
            expected = published.get('sha256')
            equal = digest == expected if expected else None
            if modo == 'arquivadas' and equal is False:
                raise ValueError('SHA-256 da cópia arquivada difere do snapshot')
            PATHS[modo][name] = path
            version = pipeline.source_version(path)
            VERSOES.append({'modo': modo, 'fonte': name, 'bytes': path.stat().st_size,
                            'sha256': digest, 'sha256_snapshot': expected, 'mesma_versao': equal,
                            'last_modified': version.get('last_modified'), 'url': url})
        except Exception as exc:
            FALHAS_DOWNLOAD.append({'modo': modo, 'fonte': name, 'erro': repr(exc)})
table(VERSOES)
table(FALHAS_DOWNLOAD)
print('Hash diferente nas fontes atuais significa mudança de versão, não prova de bug.')

Baixando/lendo: atuais changes
Baixando/lendo: atuais registry
Baixando/lendo: atuais votes
Baixando/lendo: atuais details
Baixando/lendo: arquivadas changes
Baixando/lendo: arquivadas registry
Baixando/lendo: arquivadas votes
Baixando/lendo: arquivadas details


modo,fonte,bytes,sha256,sha256_snapshot,mesma_versao,last_modified,url
atuais,changes,13577,b52b8c1d4d394ee5d0b74a950c5496f53788b99abcfaa3dfd11a2fd28227f1d2,b52b8c1d4d394ee5d0b74a950c5496f53788b99abcfaa3dfd11a2fd28227f1d2,True,"Mon, 19 Oct 2026 14:12:23 GMT",https://cdn.tse.jus.br/estatistica/sead/odsele/arquivos_gerados/eleitorado_eleicao_2026/20261019_105000/71d5e6b06fa299337b05541127926cd7/secoes_eleitorais_alteradas_2t-pais_2026.csv.zip
atuais,registry,176387413,7690af742f2d0f476353642e3b8fa73399301bdbce2b47d5a83f6085d9ec38e3,8faee1010aefc0144cd6388d3ef9076f2f218dfa998740a764e8bf05e3ec00cb,False,"Wed, 21 Oct 2026 14:34:01 GMT",https://cdn.tse.jus.br/estatistica/sead/odsele/eleitorado_locais_votacao/eleitorado_local_votacao_2026.zip
atuais,votes,161616519,2d42006d0f6c6e00bad83f5c23cd3d78cd3e3182dbc1ad6718222b6b132c23e1,2d42006d0f6c6e00bad83f5c23cd3d78cd3e3182dbc1ad6718222b6b132c23e1,True,"Tue, 06 Oct 2026 19:22:49 GMT",https://cdn.tse.jus.br/estatistica/sead/odsele/votacao_secao/votacao_secao_2026_BR.zip
atuais,details,176923175,11a62c3a604e8b73fc328617b7688d2c635de1c76df02ba63b8f77879512f573,11a62c3a604e8b73fc328617b7688d2c635de1c76df02ba63b8f77879512f573,True,"Wed, 07 Oct 2026 20:54:59 GMT",https://cdn.tse.jus.br/estatistica/sead/odsele/detalhe_votacao_secao/detalhe_votacao_secao_2026.zip
arquivadas,changes,13577,b52b8c1d4d394ee5d0b74a950c5496f53788b99abcfaa3dfd11a2fd28227f1d2,b52b8c1d4d394ee5d0b74a950c5496f53788b99abcfaa3dfd11a2fd28227f1d2,True,"Wed, 21 Oct 2026 17:44:34 GMT",https://storage.googleapis.com/extracao_alteracao_locais_votacao/fontes/changes/20261021T174431869724Z-42c52f45027d4f3387cb2f10dd621578-20261019T141223Z-b52b8c1d4d39-secoes_eleitorais_alteradas_2t-pais_2026.csv.zip
arquivadas,registry,175502489,8faee1010aefc0144cd6388d3ef9076f2f218dfa998740a764e8bf05e3ec00cb,8faee1010aefc0144cd6388d3ef9076f2f218dfa998740a764e8bf05e3ec00cb,True,"Wed, 21 Oct 2026 17:44:36 GMT",https://storage.googleapis.com/extracao_alteracao_locais_votacao/fontes/registry/20261021T174434072461Z-70c1d248eb03466fb81e9b95ac0ac261-20261020T233204Z-8faee1010aef-eleitorado_local_votacao_2026.zip
arquivadas,votes,161616519,2d42006d0f6c6e00bad83f5c23cd3d78cd3e3182dbc1ad6718222b6b132c23e1,2d42006d0f6c6e00bad83f5c23cd3d78cd3e3182dbc1ad6718222b6b132c23e1,True,"Wed, 21 Oct 2026 17:44:39 GMT",https://storage.googleapis.com/extracao_alteracao_locais_votacao/fontes/votes/20261021T174436463434Z-e1c60035104f43c0a2d4e99c00ed0aca-20261006T192249Z-2d42006d0f6c-votacao_secao_2026_BR.zip
arquivadas,details,176923175,11a62c3a604e8b73fc328617b7688d2c635de1c76df02ba63b8f77879512f573,11a62c3a604e8b73fc328617b7688d2c635de1c76df02ba63b8f77879512f573,True,"Wed, 21 Oct 2026 17:44:42 GMT",https://storage.googleapis.com/extracao_alteracao_locais_votacao/fontes/details/20261021T174439229335Z-4a22c634e72046729b2f29a32674913a-20261007T205459Z-11a62c3a604e-detalhe_votacao_secao_2026.zip


8 registro(s); exibidos até 20.
(sem registros)
Hash diferente nas fontes atuais significa mudança de versão, não prova de bug.


In [9]:
# 5. Leitor independente e normalização das chaves (sem usar iter_csv do pipeline)
def norm(value):
    text = unicodedata.normalize('NFKD', str(value).upper())
    return ''.join(c for c in text if c.isascii() and c.isalnum())

def raw_rows(path):
    def records(binary):
        with io.TextIOWrapper(binary, encoding=CONFIG.get('encoding', 'cp1252'), newline='') as text:
            reader = csv.DictReader(text, delimiter=';')
            assert reader.fieldnames, 'CSV sem cabeçalho'
            reader.fieldnames = [k.lstrip('\ufeff').strip().upper() for k in reader.fieldnames]
            for row in reader:
                assert None not in row and all(v is not None for v in row.values()), 'CSV com linha malformada'
                yield {k: v.strip() for k, v in row.items()}
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as archive:
            members = sorted(n for n in archive.namelist() if n.lower().endswith('.csv') and not n.startswith('__MACOSX/'))
            national = [n for n in members if re.search(r'_(BRASIL|BR)\.csv$', n, re.I)]
            # O ZIP de detalhes contém BR e BRASIL: escolhemos BR explicitamente,
            # nunca somamos os dois arquivos nacionais (nem os arquivos de UF).
            preferred = sorted(national, key=lambda n: (not n.upper().endswith('_BR.CSV'), n))[:1]
            for member in (preferred or members):
                with archive.open(member) as stream:
                    yield from records(stream)
    else:
        with path.open('rb') as stream:
            yield from records(stream)

def named_key(row):
    return (row['SG_UF'].upper(), norm(row['NM_MUNICIPIO']), int(row['NR_ZONA']), int(row['NR_SECAO']))

def json_key(row):
    return (row['uf'].upper(), norm(row['municipio']), int(row['zona']), int(row['secao']))

JSON_INDEX = {json_key(r): r for r in JSON_ROWS}
DUPLICADAS_JSON = [k for k, n in Counter(json_key(r) for r in JSON_ROWS).items() if n > 1]
BRUTAS = {}
for modo, paths in PATHS.items():
    if 'changes' not in paths:
        continue
    rows = [r for r in raw_rows(paths['changes']) if int(r['AA_ELEICAO']) == CONFIG['year']]
    counts = Counter(named_key(r) for r in rows)
    BRUTAS[modo] = {'changes': rows, 'index': {named_key(r): r for r in rows},
                    'duplicadas': [k for k, n in counts.items() if n > 1]}
    print(modo, '— amostra real da lista do TSE:')
    table(rows, limit=3)
print('Duplicadas JSON:', DUPLICADAS_JSON)
print('CD_MUNICIPIO do TSE NÃO é cd_ibge. A chave comparável usa UF + nome normalizado + zona + seção.')

atuais — amostra real da lista do TSE:


AA_ELEICAO,NM_PAIS,SG_UF,NM_MUNICIPIO,NR_ZONA,NM_LOCAL_VOTACAO_INICIAL,NR_SECAO,NM_LOCAL_VOTACAO_FINAL,QT_ELEITORES
2026,Brasil,AC,BRASILÉIA,6,ESCOLA FONTENELE DE CASTRO,54,CAMPUS FRONTEIRA DO ALTO ACRE - UFAC,233
2026,Brasil,AC,BRASILÉIA,6,ESCOLA FONTENELE DE CASTRO,113,ESCOLA ANTÔNIO CARNEIRO DA SILVA - KM 04 POLO AGROFLORESTAL,255
2026,Brasil,AC,BRASILÉIA,6,ESCOLA MENINO JESUS,146,UNIDADE BÁSICA DE SAÚDE SIMÃO MANSOUR BARTHA,272


774 registro(s); exibidos até 3.
arquivadas — amostra real da lista do TSE:


AA_ELEICAO,NM_PAIS,SG_UF,NM_MUNICIPIO,NR_ZONA,NM_LOCAL_VOTACAO_INICIAL,NR_SECAO,NM_LOCAL_VOTACAO_FINAL,QT_ELEITORES
2026,Brasil,AC,BRASILÉIA,6,ESCOLA FONTENELE DE CASTRO,54,CAMPUS FRONTEIRA DO ALTO ACRE - UFAC,233
2026,Brasil,AC,BRASILÉIA,6,ESCOLA FONTENELE DE CASTRO,113,ESCOLA ANTÔNIO CARNEIRO DA SILVA - KM 04 POLO AGROFLORESTAL,255
2026,Brasil,AC,BRASILÉIA,6,ESCOLA MENINO JESUS,146,UNIDADE BÁSICA DE SAÚDE SIMÃO MANSOUR BARTHA,272


774 registro(s); exibidos até 3.
Duplicadas JSON: []
CD_MUNICIPIO do TSE NÃO é cd_ibge. A chave comparável usa UF + nome normalizado + zona + seção.


In [6]:
# 6. Comparação independente: lista de alterações × JSON, registro a registro
DIVERGENCIAS = []
RESUMOS = []
AUSENTES = []
for modo, data in BRUTAS.items():
    source_index = data['index']
    missing = sorted(source_index.keys() - JSON_INDEX.keys())
    extra = sorted(JSON_INDEX.keys() - source_index.keys())
    AUSENTES += [{'modo': modo, 'lado': 'ausente_no_json', 'chave': k} for k in missing]
    AUSENTES += [{'modo': modo, 'lado': 'somente_no_json', 'chave': k} for k in extra]
    for k in source_index.keys() & JSON_INDEX.keys():
        raw, published = source_index[k], JSON_INDEX[k]
        for field, expected in {'eleitores': int(raw['QT_ELEITORES']),
                                'local_1t': raw['NM_LOCAL_VOTACAO_INICIAL'],
                                'local_2t': raw['NM_LOCAL_VOTACAO_FINAL']}.items():
            if published.get(field) != expected:
                DIVERGENCIAS.append({'modo': modo, 'etapa': 'alteracoes', 'chave': k, 'campo': field,
                                     'fonte': expected, 'json': published.get(field)})
    RESUMOS.append({'modo': modo, 'secoes_fonte': len(data['changes']), 'secoes_json': len(JSON_ROWS),
                    'eleitores_fonte': sum(int(r['QT_ELEITORES']) for r in data['changes']),
                    'eleitores_json': sum(r['eleitores'] for r in JSON_ROWS),
                    'ausentes_json': len(missing), 'extras_json': len(extra), 'duplicadas_fonte': len(data['duplicadas'])})
table(RESUMOS)
table(DIVERGENCIAS, LIMITE_AMOSTRAS)
table(AUSENTES, LIMITE_AMOSTRAS)

# Totais por UF: úteis para localizar uma divergência sem esconder compensações nacionais.
POR_UF = []
for modo, data in BRUTAS.items():
    for uf in sorted({r['SG_UF'] for r in data['changes']} | {r['uf'] for r in JSON_ROWS}):
        raw = [r for r in data['changes'] if r['SG_UF'] == uf]
        pub = [r for r in JSON_ROWS if r['uf'] == uf]
        POR_UF.append({'modo': modo, 'uf': uf, 'secoes_fonte': len(raw), 'secoes_json': len(pub),
                       'eleitores_fonte': sum(int(r['QT_ELEITORES']) for r in raw),
                       'eleitores_json': sum(r['eleitores'] for r in pub)})
table(POR_UF, 60)

modo,secoes_fonte,secoes_json,eleitores_fonte,eleitores_json,ausentes_json,extras_json,duplicadas_fonte
atuais,774,774,229018,229018,0,0,0
arquivadas,774,774,229018,229018,0,0,0


2 registro(s); exibidos até 20.
(sem registros)
(sem registros)


modo,uf,secoes_fonte,secoes_json,eleitores_fonte,eleitores_json
atuais,AC,7,7,1523,1523
atuais,AL,19,19,4860,4860
atuais,AM,48,48,16552,16552
atuais,AP,38,38,11614,11614
atuais,BA,48,48,14321,14321
atuais,CE,9,9,2381,2381
atuais,ES,7,7,1957,1957
atuais,GO,44,44,12451,12451
atuais,MA,21,21,5916,5916
atuais,MG,93,93,23919,23919


46 registro(s); exibidos até 60.


In [7]:
# 7. Cadastro bruto: correspondência de seções e ponte CD_MUNICIPIO TSE → nome
CADASTROS = {}
for modo, paths in PATHS.items():
    if modo not in BRUTAS or 'registry' not in paths:
        continue
    selected = BRUTAS[modo]['index']
    matching = defaultdict(list)
    bridge = defaultdict(set)
    for row in raw_rows(paths['registry']):
        if int(row['AA_ELEICAO']) != CONFIG['year'] or named_key(row) not in selected:
            continue
        k = named_key(row)
        matching[k].append(row)
        numeric = (row['SG_UF'], str(int(row['CD_MUNICIPIO'])), int(row['NR_ZONA']), int(row['NR_SECAO']))
        bridge[numeric].add(k)
    CADASTROS[modo] = {'rows': dict(matching), 'bridge': dict(bridge)}
    print(modo, '— seções da lista encontradas no cadastro:', len(matching), '/', len(selected))
    investigate = [r for k, rows in matching.items() if (k[0], k[2], k[3]) == SECAO_INVESTIGAR for r in rows]
    columns = ['SG_UF', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO', 'NR_TURNO', 'NR_LOCAL_VOTACAO',
               'NM_LOCAL_VOTACAO', 'DS_ENDERECO', 'NR_LATITUDE', 'NR_LONGITUDE',
               'NR_LOCAL_VOTACAO_ORIGINAL', 'NM_LOCAL_VOTACAO_ORIGINAL', 'DS_ENDERECO_LOCVT_ORIGINAL']
    table([{c: r.get(c) for c in columns} for r in investigate], LIMITE_AMOSTRAS)
print('O cadastro pode ter 1º/2º turno e locais só em *_ORIGINAL. Não somamos suas linhas como se fossem seções distintas.')

atuais — seções da lista encontradas no cadastro: 774 / 774


SG_UF,NM_MUNICIPIO,NR_ZONA,NR_SECAO,NR_TURNO,NR_LOCAL_VOTACAO,NM_LOCAL_VOTACAO,DS_ENDERECO,NR_LATITUDE,NR_LONGITUDE,NR_LOCAL_VOTACAO_ORIGINAL,NM_LOCAL_VOTACAO_ORIGINAL,DS_ENDERECO_LOCVT_ORIGINAL
PR,ALMIRANTE TAMANDARÉ,171,110,2,1627,ESCOLA MUNICIPAL ALVORADA,"RUA INDAIAL, 0, BAIRRO JARDIM APUCARANA","-25,3430183","-49,2529433",1171,ESCOLA ESTADUAL JARDIM APUCARANA,"RUA INDAIAL, 227"
PR,ALMIRANTE TAMANDARÉ,171,110,1,1627,ESCOLA MUNICIPAL ALVORADA,"RUA INDAIAL, 0, BAIRRO JARDIM APUCARANA","-25,3430183","-49,2529433",1171,ESCOLA ESTADUAL JARDIM APUCARANA,"RUA INDAIAL, 227"


2 registro(s); exibidos até 10.
arquivadas — seções da lista encontradas no cadastro: 774 / 774


SG_UF,NM_MUNICIPIO,NR_ZONA,NR_SECAO,NR_TURNO,NR_LOCAL_VOTACAO,NM_LOCAL_VOTACAO,DS_ENDERECO,NR_LATITUDE,NR_LONGITUDE,NR_LOCAL_VOTACAO_ORIGINAL,NM_LOCAL_VOTACAO_ORIGINAL,DS_ENDERECO_LOCVT_ORIGINAL
PR,ALMIRANTE TAMANDARÉ,171,110,1,1627,ESCOLA MUNICIPAL ALVORADA,"RUA INDAIAL, 0, BAIRRO JARDIM APUCARANA","-25,3430183","-49,2529433",1171,ESCOLA ESTADUAL JARDIM APUCARANA,"RUA INDAIAL, 227"
PR,ALMIRANTE TAMANDARÉ,171,110,2,1635,ESCOLA MUNICIPAL ALVORADA,"RUA INDAIAL, S/N 0",-1,-1,1171,ESCOLA ESTADUAL JARDIM APUCARANA,"RUA INDAIAL, 227"
PR,ALMIRANTE TAMANDARÉ,171,110,2,1627,ESCOLA MUNICIPAL ALVORADA,"RUA INDAIAL, 0, BAIRRO JARDIM APUCARANA","-25,3430183","-49,2529433",1171,ESCOLA ESTADUAL JARDIM APUCARANA,"RUA INDAIAL, 227"


3 registro(s); exibidos até 10.
O cadastro pode ter 1º/2º turno e locais só em *_ORIGINAL. Não somamos suas linhas como se fossem seções distintas.


In [10]:
# 8. Votos independentes: presidente (cargo 1), 1º turno, ano configurado
VOTOS_RESUMO = []
VOTOS_PROBLEMAS = []
RAW_VOTES = {}
RAW_DETAILS = {}
def numeric_key(row):
    return (row['SG_UF'], str(int(row['CD_MUNICIPIO'])), int(row['NR_ZONA']), int(row['NR_SECAO']))
def presidente_1t(row):
    return int(row['ANO_ELEICAO']) == CONFIG['year'] and int(row['NR_TURNO']) == 1 and int(row['CD_CARGO']) == 1

for modo, cadastro in CADASTROS.items():
    if not {'votes', 'details'} <= PATHS[modo].keys():
        continue
    bridge = cadastro['bridge']
    votes = {}
    details = {}
    for row in raw_rows(PATHS[modo]['votes']):
        if not presidente_1t(row):
            continue
        k = numeric_key(row)
        if k not in bridge:
            continue
        identity = (k, str(int(row['NR_VOTAVEL'])))
        value = int(row['QT_VOTOS'])
        if identity in votes and votes[identity] != value:
            VOTOS_PROBLEMAS.append({'modo': modo, 'chave': k, 'erro': 'votavel duplicado divergente'})
        votes[identity] = value
    columns = {'aptos_1t': 'QT_APTOS', 'abstencao_1t': 'QT_ABSTENCOES',
               'brancos_1t': 'QT_VOTOS_BRANCOS', 'nulos_1t': 'QT_VOTOS_NULOS'}
    for row in raw_rows(PATHS[modo]['details']):
        if not presidente_1t(row):
            continue
        k = numeric_key(row)
        if k not in bridge:
            continue
        values = {f: int(row[c]) for f, c in columns.items()}
        if k in details and details[k] != values:
            VOTOS_PROBLEMAS.append({'modo': modo, 'chave': k, 'erro': 'detalhe duplicado divergente'})
        details[k] = values
    totals = defaultdict(lambda: dict.fromkeys(['lula_1t', 'flavio_1t', 'outros_1t', 'brancos_1t', 'nulos_1t'], 0))
    for (k, code), count in votes.items():
        field = ('lula_1t' if code == str(CONFIG['candidates']['lula']) else
                 'flavio_1t' if code == str(CONFIG['candidates']['flavio']) else
                 'brancos_1t' if code == '95' else 'nulos_1t' if code == '96' else 'outros_1t')
        totals[k][field] += count
    for k, named_keys in bridge.items():
        if len(named_keys) != 1 or k not in totals or k not in details:
            VOTOS_PROBLEMAS.append({'modo': modo, 'chave': k, 'erro': 'ponte ambígua ou votação/detalhe ausente'})
            continue
        named = next(iter(named_keys))
        if named not in JSON_INDEX:
            continue
        if sum(totals[k].values()) + details[k]['abstencao_1t'] != details[k]['aptos_1t']:
            VOTOS_PROBLEMAS.append({'modo': modo, 'chave': k, 'erro': 'votos + abstenções != aptos'})
        for field in ('brancos_1t', 'nulos_1t'):
            if totals[k][field] != details[k][field]:
                VOTOS_PROBLEMAS.append({'modo': modo, 'chave': k, 'erro': field + ': votação != detalhe'})
        for field, expected in (details[k] | totals[k]).items():
            if JSON_INDEX[named].get(field) != expected:
                DIVERGENCIAS.append({'modo': modo, 'etapa': 'votos_independente', 'chave': named,
                                     'campo': field, 'fonte': expected, 'json': JSON_INDEX[named].get(field)})
    RAW_VOTES[modo] = dict(totals)
    RAW_DETAILS[modo] = details
    VOTOS_RESUMO.append({'modo': modo, 'secoes_com_votos': len(totals), 'secoes_com_detalhe': len(details),
                         'problemas': sum(r['modo'] == modo for r in VOTOS_PROBLEMAS),
                         'campos_divergentes_json': sum(r['modo'] == modo and r['etapa'] == 'votos_independente' for r in DIVERGENCIAS)})
table(VOTOS_RESUMO)
table(VOTOS_PROBLEMAS, LIMITE_AMOSTRAS)
table([r for r in DIVERGENCIAS if r['etapa'] == 'votos_independente'], LIMITE_AMOSTRAS)

modo,secoes_com_votos,secoes_com_detalhe,problemas,campos_divergentes_json
atuais,774,774,0,0
arquivadas,774,774,0,0


2 registro(s); exibidos até 20.
(sem registros)
(sem registros)


In [11]:
# 9. Reprodução do cruzamento, separada das checagens independentes
# Usa o código LOCAL, que pode diferir da imagem de um snapshot antigo.
# O IBGE é atual (não arquivado); os indicadores sem fonte não são inventados.
IBGE_PATH = pipeline.download(CONFIG['sources']['municipalities'], CACHE / 'ibge', refresh=ATUALIZAR_DOWNLOADS)
IBGE = pipeline.load_ibge(IBGE_PATH, CONFIG.get('ibgeOverrides', {}))
IBGE_HASH = sha256_file(IBGE_PATH)
print('IBGE tem mesmo hash do snapshot:', IBGE_HASH == VERSOES_SNAPSHOT.get('municipalities', {}).get('sha256'))
REPRODUCOES = {}
DIFERENCAS_REPRODUCAO = []
FALHAS_REPRODUCAO = []
for modo, paths in PATHS.items():
    if not set(FONTES_TSE) <= paths.keys():
        FALHAS_REPRODUCAO.append({'modo': modo, 'erro': 'Conjunto TSE incompleto'})
        continue
    if CONFIG['sources'].get('longDistance'):
        FALHAS_REPRODUCAO.append({'modo': modo, 'erro': 'Fonte longDistance configurada: adaptar reprodução antes de comparar'})
        continue
    try:
        changes = pipeline.load_changes(paths['changes'], CONFIG['encoding'], CONFIG['year'])
        sections, locations, names, places = pipeline.load_registry(paths['registry'], changes, CONFIG['encoding'], CONFIG['year'], allow_incomplete=True)
        wanted = {next(iter(names[k])) for k in changes if len(names.get(k, set())) == 1}
        results = pipeline.load_results(paths['votes'], paths['details'], wanted, CONFIG['encoding'], CONFIG['year'],
                                        str(CONFIG['candidates']['lula']), str(CONFIG['candidates']['flavio']))
        rows, issues = pipeline.build_rows(changes, sections, locations, names, IBGE, results, {}, places)
        rebuilt = {json_key(r): typed_row(r, pipeline.FIELDS) for r in rows}
        REPRODUCOES[modo] = {'rows': rebuilt, 'pendencias': issues}
        for k in rebuilt.keys() | JSON_INDEX.keys():
            if k not in rebuilt or k not in JSON_INDEX:
                DIFERENCAS_REPRODUCAO.append({'modo': modo, 'chave': k, 'campo': '__registro__', 'fonte': k in rebuilt, 'json': k in JSON_INDEX})
                continue
            for field in pipeline.FIELDS:
                a, b = rebuilt[k].get(field), JSON_INDEX[k].get(field)
                equal = math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-6) if type(a) in (int, float) and type(b) in (int, float) else a == b
                if not equal:
                    DIFERENCAS_REPRODUCAO.append({'modo': modo, 'chave': k, 'campo': field, 'fonte': a, 'json': b})
        print(modo, '— linhas:', len(rebuilt), 'pendências:', len(issues))
        table([{'pendencia': issue} for issue in issues], limit=5)
    except Exception as exc:
        FALHAS_REPRODUCAO.append({'modo': modo, 'erro': repr(exc)})
table(DIFERENCAS_REPRODUCAO, LIMITE_AMOSTRAS)
table(FALHAS_REPRODUCAO)
print('Divergências com fonte atual de hash diferente são mudanças observadas, não prova de bug da extração anterior.')

IBGE tem mesmo hash do snapshot: True
arquivadas — linhas: 774 pendências: 425


pendencia
"('AC', 'BRASILEIA', 6, 54): local final só nas colunas originais do cadastro; sem coordenadas disponíveis"
"('AC', 'BRASILEIA', 6, 113): local final só nas colunas originais do cadastro; sem coordenadas disponíveis"
"('AC', 'BRASILEIA', 6, 146): local final só nas colunas originais do cadastro; sem coordenadas disponíveis"
"('AC', 'BRASILEIA', 6, 165): local final só nas colunas originais do cadastro; sem coordenadas disponíveis"
"('AC', 'CAPIXABA', 2, 88): local final só nas colunas originais do cadastro; sem coordenadas disponíveis"


425 registro(s); exibidos até 5.
(sem registros)


modo,erro
atuais,"DataError(""Local ambíguo no cadastro: ('MG', 'AIMORES', 5, 4) inicial; não é seguro escolher pelas coordenadas"")"


1 registro(s); exibidos até 20.
Divergências com fonte atual de hash diferente são mudanças observadas, não prova de bug da extração anterior.


In [12]:
# 10. Schema e invariantes do JSON, sem converter tipos inválidos silenciosamente
INTEGER_FIELDS = {'cd_ibge', 'zona', 'secao', 'eleitores', 'aptos_1t', 'abstencao_1t',
                  'lula_1t', 'flavio_1t', 'outros_1t', 'brancos_1t', 'nulos_1t'}
FLOAT_FIELDS = {'lat_1t', 'lon_1t', 'lat_2t', 'lon_2t', 'distancia_km'}
NON_NULL = {'uf', 'municipio', 'cd_ibge', 'zona', 'secao', 'eleitores', 'local_1t', 'local_2t'}
SCHEMA_ERROS = []
INVARIANTES_ERROS = []
for i, row in enumerate(JSON_ROWS):
    for field in pipeline.FIELDS:
        if field not in row:
            SCHEMA_ERROS.append({'linha': i, 'campo': field, 'erro': 'campo ausente'})
            continue
        value = row[field]
        if value is None:
            if field in NON_NULL:
                SCHEMA_ERROS.append({'linha': i, 'campo': field, 'erro': 'null obrigatório'})
            continue
        valid = (type(value) is int and value >= 0 if field in INTEGER_FIELDS else
                 type(value) in (int, float) and math.isfinite(value) if field in FLOAT_FIELDS else
                 type(value) is bool if field == 'regiao_longe_da_urna' else isinstance(value, str))
        if not valid:
            SCHEMA_ERROS.append({'linha': i, 'campo': field, 'valor': value, 'erro': 'tipo/valor inválido'})
    for turn in ('1t', '2t'):
        lat, lon = row.get('lat_' + turn), row.get('lon_' + turn)
        if (lat is None) != (lon is None):
            INVARIANTES_ERROS.append({'linha': i, 'erro': 'par de coordenadas incompleto', 'turno': turn})
        if lat is not None and (type(lat) not in (int, float) or not math.isfinite(lat) or abs(lat) > 90):
            INVARIANTES_ERROS.append({'linha': i, 'erro': 'latitude inválida', 'turno': turn})
        if lon is not None and (type(lon) not in (int, float) or not math.isfinite(lon) or abs(lon) > 180):
            INVARIANTES_ERROS.append({'linha': i, 'erro': 'longitude inválida', 'turno': turn})
    fields = ['lula_1t', 'flavio_1t', 'outros_1t', 'brancos_1t', 'nulos_1t', 'abstencao_1t']
    if all(type(row.get(f)) is int for f in fields + ['aptos_1t']):
        if sum(row[f] for f in fields) != row['aptos_1t']:
            INVARIANTES_ERROS.append({'linha': i, 'erro': 'votos + abstenções != aptos'})
    if row.get('distancia_km') is not None and type(row['distancia_km']) in (int, float) and row['distancia_km'] < 0:
        INVARIANTES_ERROS.append({'linha': i, 'erro': 'distância negativa'})

NULOS = [{'campo': f, 'nulos': sum(r.get(f) is None for r in JSON_ROWS), 'total': len(JSON_ROWS)} for f in pipeline.FIELDS]
table(SCHEMA_ERROS, LIMITE_AMOSTRAS)
table(INVARIANTES_ERROS, LIMITE_AMOSTRAS)
table(NULOS, 30)
print('Null de coordenadas pode refletir ausência/ambiguidade de fonte. Longe da urna não equivale à distância da mudança.')

(sem registros)
(sem registros)


campo,nulos,total
uf,0,774
municipio,0,774
cd_ibge,0,774
zona,0,774
secao,0,774
eleitores,0,774
local_1t,0,774
endereco_1t,45,774
lat_1t,256,774
lon_1t,256,774


24 registro(s); exibidos até 30.
Null de coordenadas pode refletir ausência/ambiguidade de fonte. Longe da urna não equivale à distância da mudança.


In [13]:
# 11. Investigação de uma seção e resumo de divergências por campo
print('Seção selecionada (UF, zona, seção):', SECAO_INVESTIGAR)
table([r for r in JSON_ROWS if (r['uf'], r['zona'], r['secao']) == SECAO_INVESTIGAR], 10)
for modo, data in BRUTAS.items():
    print('Lista TSE:', modo)
    table([r for k, r in data['index'].items() if (k[0], k[2], k[3]) == SECAO_INVESTIGAR], 10)
    if modo in REPRODUCOES:
        print('Cruzamento reproduzido:', modo)
        table([r for k, r in REPRODUCOES[modo]['rows'].items() if (k[0], k[2], k[3]) == SECAO_INVESTIGAR], 10)

CONTAGEM_DIFERENCAS = [{'modo': modo, 'etapa': etapa, 'campo': campo, 'diferencas': count}
    for (modo, etapa, campo), count in Counter(
        (d['modo'], d.get('etapa', 'reproducao'), d['campo'])
        for d in DIVERGENCIAS + DIFERENCAS_REPRODUCAO).items()]
table(CONTAGEM_DIFERENCAS, 100)
print('Compare a amostra acima com as linhas brutas do cadastro na etapa 7. Não escolha coordenadas de outra zona só por nome parecido.')

Seção selecionada (UF, zona, seção): ('PR', 171, 110)


uf,municipio,cd_ibge,zona,secao,eleitores,local_1t,endereco_1t,lat_1t,lon_1t,local_2t,endereco_2t,bairro_2t,lat_2t,lon_2t,distancia_km,regiao_longe_da_urna,aptos_1t,abstencao_1t,lula_1t,flavio_1t,outros_1t,brancos_1t,nulos_1t
PR,ALMIRANTE TAMANDARÉ,4100400,171,110,358,ESCOLA MUNICIPAL ALVORADA,None,None,None,ESCOLA MUNICIPAL ALVORADA,None,None,None,None,None,None,357,54,99,154,30,9,11


1 registro(s); exibidos até 10.
Lista TSE: atuais


AA_ELEICAO,NM_PAIS,SG_UF,NM_MUNICIPIO,NR_ZONA,NM_LOCAL_VOTACAO_INICIAL,NR_SECAO,NM_LOCAL_VOTACAO_FINAL,QT_ELEITORES
2026,Brasil,PR,ALMIRANTE TAMANDARÉ,171,ESCOLA MUNICIPAL ALVORADA,110,ESCOLA MUNICIPAL ALVORADA,358


1 registro(s); exibidos até 10.
Lista TSE: arquivadas


AA_ELEICAO,NM_PAIS,SG_UF,NM_MUNICIPIO,NR_ZONA,NM_LOCAL_VOTACAO_INICIAL,NR_SECAO,NM_LOCAL_VOTACAO_FINAL,QT_ELEITORES
2026,Brasil,PR,ALMIRANTE TAMANDARÉ,171,ESCOLA MUNICIPAL ALVORADA,110,ESCOLA MUNICIPAL ALVORADA,358


1 registro(s); exibidos até 10.
Cruzamento reproduzido: arquivadas


uf,municipio,cd_ibge,zona,secao,eleitores,local_1t,endereco_1t,lat_1t,lon_1t,local_2t,endereco_2t,bairro_2t,lat_2t,lon_2t,distancia_km,regiao_longe_da_urna,aptos_1t,abstencao_1t,lula_1t,flavio_1t,outros_1t,brancos_1t,nulos_1t
PR,ALMIRANTE TAMANDARÉ,4100400,171,110,358,ESCOLA MUNICIPAL ALVORADA,None,None,None,ESCOLA MUNICIPAL ALVORADA,None,None,None,None,None,None,357,54,99,154,30,9,11


1 registro(s); exibidos até 10.
(sem registros)
Compare a amostra acima com as linhas brutas do cadastro na etapa 7. Não escolha coordenadas de outra zona só por nome parecido.


In [14]:
# 12. Relatório por etapa — falha/incompleto nunca vira aprovação silenciosa
ETAPAS = []
def etapa(name, status, quantidade, detalhe):
    ETAPAS.append({'etapa': name, 'status': status, 'quantidade': quantidade, 'detalhe': detalhe})

etapa('schema_json', 'FALHOU' if SCHEMA_ERROS else 'OK', len(SCHEMA_ERROS), 'Campos obrigatórios e tipos')
etapa('invariantes_json', 'FALHOU' if INVARIANTES_ERROS else 'OK', len(INVARIANTES_ERROS), 'Coordenadas e fechamento de votos')
etapa('duplicadas_json', 'FALHOU' if DUPLICADAS_JSON else 'OK', len(DUPLICADAS_JSON), 'UF + município + zona + seção')
for modo in PATHS:
    versions = [v for v in VERSOES if v['modo'] == modo]
    same = len(versions) == 4 and all(v['mesma_versao'] is True for v in versions)
    download_errors = [e for e in FALHAS_DOWNLOAD if e['modo'] == modo]
    etapa(modo + ':downloads', 'INCOMPLETO' if download_errors or len(versions) != 4 else 'OK', len(download_errors), 'Quatro fontes obtidas')
    etapa(modo + ':versao', 'OK' if same else 'ATENÇÃO', sum(v['mesma_versao'] is not True for v in versions), 'SHA-256 comparado ao snapshot; diferente/desconhecido não prova bug')
    changes_errors = sum(d['modo'] == modo and d['etapa'] == 'alteracoes' for d in DIVERGENCIAS) + sum(d['modo'] == modo for d in AUSENTES)
    changes_errors += len(BRUTAS.get(modo, {}).get('duplicadas', []))
    etapa(modo + ':alteracoes', 'INCOMPLETO' if modo not in BRUTAS else ('DIVERGENTE' if changes_errors else 'OK'), changes_errors, 'Chaves, eleitores e nomes dos locais')
    votes_errors = sum(d['modo'] == modo and d['etapa'] == 'votos_independente' for d in DIVERGENCIAS) + sum(d['modo'] == modo for d in VOTOS_PROBLEMAS)
    etapa(modo + ':votos', 'INCOMPLETO' if modo not in RAW_VOTES else ('DIVERGENTE' if votes_errors else 'OK'), votes_errors, 'Leitura independente de votação e detalhe')
    repro_errors = sum(d['modo'] == modo for d in DIFERENCAS_REPRODUCAO)
    failures = [e['erro'] for e in FALHAS_REPRODUCAO if e['modo'] == modo]
    etapa(modo + ':reproducao', 'INCOMPLETO' if failures or modo not in REPRODUCOES else ('DIVERGENTE' if repro_errors else 'OK'), repro_errors, '; '.join(failures) or 'Código local aplicado às fontes; não é checagem independente')
    if modo in REPRODUCOES:
        issues = REPRODUCOES[modo]['pendencias']
        etapa(modo + ':cobertura', 'ATENÇÃO' if issues else 'OK', len(issues), 'Pendências de enriquecimento/ambiguidade; ver lista completa no relatório')

STATUS_GERAL = 'REQUER REVISÃO' if any(e['status'] != 'OK' for e in ETAPAS) else 'OK NAS VERIFICAÇÕES EXECUTADAS'
REPORT = {'run_id': RUN_ID, 'snapshot': SNAPSHOT['extracted_at'], 'json_url': JSON_URL,
          'json_sha256': hashlib.sha256(JSON_BYTES).hexdigest(), 'ibge_sha256': IBGE_HASH,
          'codigo_local_sha256': sha256_file(Path(pipeline.__file__)),
          'config': CONFIG, 'status': STATUS_GERAL, 'etapas': ETAPAS, 'versoes': VERSOES,
          'falhas_download': FALHAS_DOWNLOAD, 'falhas_reproducao': FALHAS_REPRODUCAO,
          'schema_erros': SCHEMA_ERROS, 'invariantes_erros': INVARIANTES_ERROS,
          'votos_problemas': VOTOS_PROBLEMAS, 'ausentes': AUSENTES,
          'divergencias_independentes': DIVERGENCIAS, 'divergencias_reproducao': DIFERENCAS_REPRODUCAO,
          'pendencias': {modo: data['pendencias'] for modo, data in REPRODUCOES.items()}}
table(ETAPAS, 100)
print('STATUS GERAL:', STATUS_GERAL)
print('OK significa apenas as verificações listadas; não certifica cobertura completa de todos os endereços/coordenadas.')

etapa,status,quantidade,detalhe
schema_json,OK,0,Campos obrigatórios e tipos
invariantes_json,OK,0,Coordenadas e fechamento de votos
duplicadas_json,OK,0,UF + município + zona + seção
atuais:downloads,OK,0,Quatro fontes obtidas
atuais:versao,ATENÇÃO,1,SHA-256 comparado ao snapshot; diferente/desconhecido não prova bug
atuais:alteracoes,OK,0,"Chaves, eleitores e nomes dos locais"
atuais:votos,OK,0,Leitura independente de votação e detalhe
atuais:reproducao,INCOMPLETO,0,"DataError(""Local ambíguo no cadastro: ('MG', 'AIMORES', 5, 4) inicial; não é seguro escolher pelas coordenadas"")"
arquivadas:downloads,OK,0,Quatro fontes obtidas
arquivadas:versao,OK,0,SHA-256 comparado ao snapshot; diferente/desconhecido não prova bug


14 registro(s); exibidos até 100.
STATUS GERAL: REQUER REVISÃO
OK significa apenas as verificações listadas; não certifica cobertura completa de todos os endereços/coordenadas.


In [15]:
# 13. Salvar diagnósticos APENAS no disco local
(OUTPUT / 'relatorio.json').write_text(json.dumps(REPORT, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
def export_csv(name, records):
    records = list(records)
    columns = list(dict.fromkeys(k for r in records for k in r)) or ['sem_registros']
    with (OUTPUT / name).open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=columns, delimiter=';')
        writer.writeheader()
        for row in records:
            writer.writerow({k: json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list, tuple)) else v for k, v in row.items()})
export_csv('etapas.csv', ETAPAS)
export_csv('versoes.csv', VERSOES)
export_csv('divergencias.csv', DIVERGENCIAS + DIFERENCAS_REPRODUCAO)
export_csv('ausentes.csv', AUSENTES)
export_csv('totais_por_uf.csv', POR_UF)
export_csv('schema_erros.csv', SCHEMA_ERROS + INVARIANTES_ERROS)
export_csv('nulos_por_campo.csv', NULOS)
export_csv('pendencias.csv', [{'modo': modo, 'pendencia': p} for modo, data in REPRODUCOES.items() for p in data['pendencias']])
print('Diagnósticos salvos em:', OUTPUT)
print('Nenhuma escrita no Cloud Storage. Reexecute a partir dos parâmetros para capturar outro instante.')

Diagnósticos salvos em: /Users/Vinicius/repos/secoes_2T2026/.cache/sanidade/relatorios/20261021T175212280869Z
Nenhuma escrita no Cloud Storage. Reexecute a partir dos parâmetros para capturar outro instante.


## Como usar e interpretar
1. Selecione o kernel `.venv` e execute as células de cima para baixo.
2. Na **célula 3**, ajuste `SNAPSHOT_AT`, `MODO`, `ATUALIZAR_DOWNLOADS` e `SECAO_INVESTIGAR`. Use `arquivadas` para reproduzir a extração; use `atuais` para observar o que o TSE serve agora.
3. A amostra do cadastro mostra registros reais por turno e campos originais. Os votos são filtrados para presidente no primeiro turno. Totais nacionais iguais não substituem a comparação registro a registro.
4. `INCOMPLETO` significa que a etapa não pôde terminar; não equivale a zero divergências. `ATENÇÃO` sinaliza versão diferente ou cobertura parcial.
5. Os diagnósticos completos ficam na pasta de saída local impressa na última célula. Evite comparar relatórios de instantes diferentes como se fossem uma única execução.

### Resultado da execução inicial (21/10/2026)
- Snapshot selecionado: 17:44:31.855417Z. Foram conferidos 774 registros e 229.018 eleitores, sem divergências independentes de alterações/votos nem erros de schema.
- A reprodução com as quatro cópias arquivadas coincidiu em todos os campos, com 425 pendências de enriquecimento registradas.
- O cadastro **atual** do TSE já tem outro SHA-256. A reprodução com ele encontrou local ambíguo em MG/AIMORÉS, zona 5, seção 4, e foi marcada como incompleta. Isso pede investigação, mas não invalida a reprodução do snapshot com suas próprias fontes.
- O notebook não altera o pipeline para contornar ambiguidades nem publica dados.